# Roman Space Telescope: Pointing Model Tutorial

This notebook is an educational walkthrough of the Roman Space Telescope pointing model based on the report *Nancy Grace Roman Observatory Pointing Angles* by Savransky (2024-04-08).

It introduces reference frames, pointing angle conventions, zero-point alignment, and how to compute pointing angles to align Roman's boresight with a target on the celestial sphere.

## 1. Nomenclature and Conventions
- **Vectors** are denoted by lowercase bold letters (e.g. **r**)
- **Unit vectors** have a hat (e.g. **r̂**)
- **Norm**: ∥**r**∥, such that **r** = ∥**r**∥·**r̂**
- **Reference frames**: `I`, `B` represent inertial and body frames respectively
- **DCM (Direction Cosine Matrix)**: Transforms vector components between frames
- **Projections**: a⊥b̂ = a − (a · b̂)·b̂
- **Angle** between unit vectors **x̂** and **ŷ** about axis **ẑ**:
  ```
  θ = atan2(∥x̂ × ŷ∥·sign(det([x̂ ŷ ẑ])), x̂ · ŷ)
  ```

## 2. Reference Frame Definitions
- **Inertial Frame I**: Barycentric ecliptic, with ê₁ (toward vernal equinox), ê₂ in ecliptic plane, ê₃ to ecliptic north.
- **Body Frame B** (Roman):
  - x̂: along telescope boresight
  - ŷ: solar panel axis
  - ẑ: orthogonal to solar panels (initially pointed at Sun)


## 3. Pointing Angle Definition
Roman uses 3 sequential rotations about body axes:
- **Yaw (ψ)**: rotate about ẑ
- **Pitch (θ)**: then about ŷ
- **Roll (ϕ)**: then about x̂

The total transformation is: `C1(ϕ)·C2(θ)·C3(ψ)`


## 4. Zero-Point Orientation
The initial orientation aligns **ẑ** (solar panel normal) with the Sun:
```python
r_sun_obs = r_sun_G - r_obs_G
```
We compute:
- `α1`: rotation about ŷ to bring ê₃ to the projection of r⊙/R onto ê₁−ê₃
- `α2`: rotation about x̂ to align ẑ with r⊙/R

DCM from I to zero-point orientation: `BCI₀ = C1(α2)·C2(α1)`

## 5. Target Pointing
Given a target (**r⋆/O**), compute the look vector:
```python
r_target_obs = r_target_G - r_obs_G
```
- Project onto x̂−ŷ plane and compute **yaw** (ψ)
- Rotate about ŷ to compute **pitch** (θ) to align boresight


## 6. Roll-Pitch Coupling
- Pitch aligns boresight with target
- Roll (ϕ) is unconstrained for pointing, but affects solar panel alignment
- Roll of ±13° allowed to manage thermal balance


In [1]:
import numpy as np
import astropy.units as u
from astropy.time import Time
from astropy.coordinates import SkyCoord, get_body_barycentric, BarycentricMeanEcliptic
import scipy.optimize
from keplertools.angutils import projplane, calcang, rotMat

In [2]:
# Compute L2 location
f = lambda x, mustar: x - (1 - mustar) * (x + mustar) / np.abs(x + mustar) ** 3 - mustar * (x - 1 + mustar) / np.abs(x - 1 + mustar) ** 3
mustar_sunearth = ((1 * u.Mearth) / (1 * u.Mearth + 1 * u.Msun)).decompose().value
fsunearth = lambda x: f(x, mustar_sunearth)
L2loc = scipy.optimize.fsolve(fsunearth, 1)[0]

In [3]:
def getSunPositions(ts):
    sun = SkyCoord(get_body_barycentric("Sun", ts), frame="icrs", obstime=ts).transform_to(BarycentricMeanEcliptic)
    return sun.cartesian.xyz

def getL2Positions(ts):
    earth = SkyCoord(get_body_barycentric("Earth", ts), frame="icrs", obstime=ts).transform_to(BarycentricMeanEcliptic)
    return L2loc * earth.cartesian.xyz

In [4]:
def calcRomanAngles(target, ts, r_obs_G, r_sun_G=None):
    from astropy.coordinates import SkyCoord, BarycentricMeanEcliptic
    import numpy as np
    import astropy.units as u

    if r_sun_G is None:
        r_sun_G = getSunPositions(ts)

    r_sun_obs = r_sun_G - r_obs_G
    rhat_sun_obs = (r_sun_obs / np.linalg.norm(r_sun_obs, axis=0)).value

    # Attempt to propagate motion
    try:
        target_updated = target.apply_space_motion(new_obstime=ts)
    except ValueError:
        target_updated = target

    # Use distance if available, otherwise set default
    if hasattr(target_updated, 'distance') and target_updated.distance is not None:
        distance = target_updated.distance
    else:
        distance = 1 * u.pc

    # Transform target to BarycentricMeanEcliptic with valid distance
    r_target_G = SkyCoord(
        ra=target_updated.icrs.ra,
        dec=target_updated.icrs.dec,
        distance=distance,
        frame="icrs",
        obstime=ts
    ).transform_to(BarycentricMeanEcliptic()).cartesian.xyz

    # Fallback: ensure result has length units
    if not hasattr(r_target_G, 'unit') or r_target_G.unit == u.dimensionless_unscaled:
        r_target_G = r_target_G * distance.to(u.AU) / distance

    r_target_obs = r_target_G - r_obs_G
    rhat_target_obs = (r_target_obs / np.linalg.norm(r_target_obs, axis=0)).value

    sun_ang = (
        np.arccos([np.dot(x, y) for x, y in zip(rhat_sun_obs.T, rhat_target_obs.T)])
        * u.rad
    )

    e2 = np.array([0, 1, 0])
    e3 = np.array([0, 0, 1])

    r_sun_obs_proj1 = projplane(r_sun_obs, e2)
    rhat_sun_obs_proj1 = (
        r_sun_obs_proj1 / np.linalg.norm(r_sun_obs_proj1, axis=0)
    ).value
    ang1 = np.array([calcang(x, e3, e2) for x in rhat_sun_obs_proj1.T])
    B_C_I = np.dstack([rotMat(2, -a) for a in ang1])

    b_3 = B_C_I[2, :, :].T
    b_1 = B_C_I[0, :, :].T
    ang2 = np.array([calcang(x, b3, b1) for x, b3, b1 in zip(rhat_sun_obs.T, b_3, b_1)])

    B_C_I = np.dstack(
        [np.matmul(rotMat(1, -a), B_C_I[:, :, j]) for j, a in enumerate(ang2)]
    )

    r_target_obs_proj1 = np.hstack(
        [
            projplane(np.array(r_target_obs[:, j], ndmin=2).T, B_C_I[2, :, j].T)
            for j in range(len(ts))
        ]
    )
    rhat_target_obs_proj1 = r_target_obs_proj1 / np.linalg.norm(
        r_target_obs_proj1, axis=0
    )

    b_1 = B_C_I[0, :, :].T
    b_3 = B_C_I[2, :, :].T
    yaw = -np.array(
        [calcang(x, b1, b3) for x, b1, b3 in zip(rhat_target_obs_proj1.T, b_1, b_3)]
    )

    B_C_I = np.dstack(
        [np.matmul(rotMat(3, a), B_C_I[:, :, j]) for j, a in enumerate(yaw)]
    )

    b_1 = B_C_I[0, :, :].T
    b_2 = B_C_I[1, :, :].T
    pitch = -np.array(
        [calcang(x, b1, b2) for x, b1, b2 in zip(rhat_target_obs.T, b_1, b_2)]
    )

    B_C_I = np.dstack(
        [np.matmul(rotMat(2, a), B_C_I[:, :, j]) for j, a in enumerate(pitch)]
    )

    return sun_ang, yaw * u.rad, pitch * u.rad, B_C_I


In [5]:
# Example: align to a target
ts = Time("2025-01-01T00:00:00", scale="tdb")
target = SkyCoord(ra=100*u.deg, dec=45*u.deg, frame="icrs")
r_obs_G = getL2Positions(ts)
r_sun_G = getSunPositions(ts)
sun_ang, yaw, pitch, B_C_I = calcRomanAngles(target, ts, r_obs_G, r_sun_G)
sun_ang.to(u.deg), yaw.to(u.deg), pitch.to(u.deg)

UnitConversionError: '' (dimensionless) and 'AU' (length) are not convertible

### Summary
- We walked through the Roman pointing model step by step
- Applied the full DCM rotation sequence
- Illustrated target alignment and sun-angle computation

See the report by Savransky (2024) for deeper derivations and diagrams.